Version: 02.14.2023

# Capstone Project: Bringing It All Together

In this lab, you will bring together many of the tools and techniques that you have learned throughout this course into a final project. You can choose from many different paths to get to the solution. You could use AWS Managed Services, such as Amazon Comprehend, or use the Amazon SageMaker models. Have fun on whichever path you choose.

### Business scenario

You work for a training organization that recently developed an introductory course about machine learning (ML). The course includes more than 40 videos that cover a broad range of ML topics. You have been asked to create an application that will students can use to quickly locate and view video content by searching for topics and key phrases.

You have downloaded all of the videos to an Amazon Simple Storage Service (Amazon S3) bucket. Your assignment is to produce a dashboard that meets your supervisor’s requirements.

To assist you, all of the previous labs have been provided in this workspace.

## Lab steps

To complete this lab, you will follow these steps:

1. [Viewing the video files](#1.-Viewing-the-video-files)
2. [Transcribing the videos](#2.-Transcribing-the-videos)
3. [Normalizing the text](#3.-Normalizing-the-text)
4. [Extracting key phrases and topics](#4.-Extracting-key-phrases-and-topics)
5. [Creating the dashboard](#5.-Creating-the-dashboard)

## Submitting your work

1. In the lab console, choose **Submit** to record your progress and when prompted, choose **Yes**.

1. If the results don't display after a couple of minutes, return to the top of these instructions and choose **Grades**.

     **Tip**: You can submit your work multiple times. After you change your work, choose **Submit** again. Your last submission is what will be recorded for this lab.

1. To find detailed feedback on your work, choose **Details** followed by **View Submission Report**.

## Useful information

The following cell contains some information that might be useful as you complete this project.

In [ ]:
bucket = "<your bucket>"
job_data_access_role = 'arn:aws:iam::<your_role>'

## 1. Viewing the video files
([Go to top](#Capstone-8:-Bringing-It-All-Together))


The source video files are located in the following shared Amazon Simple Storage Service (Amazon S3) bucket.

In [18]:
!aws s3 ls s3://aws-tc-largeobjects/CUR-TF-200-ACMNLP-1/video/

2021-04-26 20:17:33  410925369 Mod01_Course Overview.mp4
2021-04-26 20:10:02   39576695 Mod02_Intro.mp4
2021-04-26 20:31:23  302994828 Mod02_Sect01.mp4
2021-04-26 20:17:33  416563881 Mod02_Sect02.mp4
2021-04-26 20:17:33  318685583 Mod02_Sect03.mp4
2021-04-26 20:17:33  255877251 Mod02_Sect04.mp4
2021-04-26 20:23:51   99988046 Mod02_Sect05.mp4
2021-04-26 20:24:54   50700224 Mod02_WrapUp.mp4
2021-04-26 20:26:27   60627667 Mod03_Intro.mp4
2021-04-26 20:26:28  272229844 Mod03_Sect01.mp4
2021-04-26 20:27:06  309127124 Mod03_Sect02_part1.mp4
2021-04-26 20:27:06  195635527 Mod03_Sect02_part2.mp4
2021-04-26 20:28:03  123924818 Mod03_Sect02_part3.mp4
2021-04-26 20:31:28  171681915 Mod03_Sect03_part1.mp4
2021-04-26 20:32:07  285200083 Mod03_Sect03_part2.mp4
2021-04-26 20:33:17  105470345 Mod03_Sect03_part3.mp4
2021-04-26 20:35:10  157185651 Mod03_Sect04_part1.mp4
2021-04-26 20:36:27  187435635 Mod03_Sect04_part2.mp4
2021-04-26 20:36:40  280720369 Mod03_Sect04_part3.mp4
2021-04-26 20:40:01  443479

## 2. Transcribing the videos
 ([Go to top](#Capstone-8:-Bringing-It-All-Together))

Use this section to implement your solution to transcribe the videos.

In [20]:
!aws s3 cp s3://aws-tc-largeobjects/CUR-TF-200-ACMNLP-1/video/ s3://{bucket}/input/ --recursive

copy: s3://aws-tc-largeobjects/CUR-TF-200-ACMNLP-1/video/Mod02_Sect01.mp4 to s3://c193754a4977429l14270447t1w992382636323-labbucket-8ksrrtliy6u4/input/Mod02_Sect01.mp4
copy: s3://aws-tc-largeobjects/CUR-TF-200-ACMNLP-1/video/Mod02_Sect03.mp4 to s3://c193754a4977429l14270447t1w992382636323-labbucket-8ksrrtliy6u4/input/Mod02_Sect03.mp4
copy: s3://aws-tc-largeobjects/CUR-TF-200-ACMNLP-1/video/Mod02_Sect05.mp4 to s3://c193754a4977429l14270447t1w992382636323-labbucket-8ksrrtliy6u4/input/Mod02_Sect05.mp4
copy: s3://aws-tc-largeobjects/CUR-TF-200-ACMNLP-1/video/Mod02_WrapUp.mp4 to s3://c193754a4977429l14270447t1w992382636323-labbucket-8ksrrtliy6u4/input/Mod02_WrapUp.mp4
copy: s3://aws-tc-largeobjects/CUR-TF-200-ACMNLP-1/video/Mod02_Intro.mp4 to s3://c193754a4977429l14270447t1w992382636323-labbucket-8ksrrtliy6u4/input/Mod02_Intro.mp4
copy: s3://aws-tc-largeobjects/CUR-TF-200-ACMNLP-1/video/Mod03_Intro.mp4 to s3://c193754a4977429l14270447t1w992382636323-labbucket-8ksrrtliy6u4/input/Mod03_Intro.

In [21]:
from boto3 import client

conn = client('s3') 
for key in conn.list_objects(Bucket=bucket)['Contents']:
    print(key['Key'])

input/Mod01_Course Overview.mp4
input/Mod02_Intro.mp4
input/Mod02_Sect01.mp4
input/Mod02_Sect02.mp4
input/Mod02_Sect03.mp4
input/Mod02_Sect04.mp4
input/Mod02_Sect05.mp4
input/Mod02_WrapUp.mp4
input/Mod03_Intro.mp4
input/Mod03_Sect01.mp4
input/Mod03_Sect02_part1.mp4
input/Mod03_Sect02_part2.mp4
input/Mod03_Sect02_part3.mp4
input/Mod03_Sect03_part1.mp4
input/Mod03_Sect03_part2.mp4
input/Mod03_Sect03_part3.mp4
input/Mod03_Sect04_part1.mp4
input/Mod03_Sect04_part2.mp4
input/Mod03_Sect04_part3.mp4
input/Mod03_Sect05.mp4
input/Mod03_Sect06.mp4
input/Mod03_Sect07_part1.mp4
input/Mod03_Sect07_part2.mp4
input/Mod03_Sect07_part3.mp4
input/Mod03_Sect08.mp4
input/Mod03_WrapUp.mp4
input/Mod04_Intro.mp4
input/Mod04_Sect01.mp4
input/Mod04_Sect02_part1.mp4
input/Mod04_Sect02_part2.mp4
input/Mod04_Sect02_part3.mp4
input/Mod04_WrapUp.mp4
input/Mod05_Intro.mp4
input/Mod05_Sect01_ver2.mp4
input/Mod05_Sect02_part1_ver2.mp4
input/Mod05_Sect02_part2.mp4
input/Mod05_Sect03_part1.mp4
input/Mod05_Sect03_part2.m

In [22]:
import boto3
import os, io, struct, json
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import uuid
from time import sleep
import re
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from nltk.tokenize import RegexpTokenizer
from nltk.stem.wordnet import WordNetLemmatizer

[nltk_data] Downloading package punkt to /home/ec2-user/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/ec2-user/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/ec2-user/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /home/ec2-user/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [14]:
# transcribe_client.list_transcription_jobs()

In [47]:
transcribe_client = boto3.client("transcribe")
s3_client = boto3.client('s3')

In [48]:
import boto3
import time
from time import sleep

In [52]:

def do_transcription_job(transcribe_job_name, media_input_uri, bucket, transcribe_output_filename):
    response = transcribe_client.start_transcription_job(
        TranscriptionJobName=transcribe_job_name,
        Media={'MediaFileUri': media_input_uri},
        MediaFormat='mp4',
        LanguageCode='en-US',
        OutputBucketName=bucket,
        OutputKey=transcribe_output_filename
    )

    

In [53]:
# implementing solution with amazon transcribe
# Avoid doing job transcription one by one over the videos waiting every job is finished
# First, doing job of all videos and then pulling all together :)
#create input paramters for job_name and job_uri
# loop on every video file listed

# Inicializar clientes

output_files=[]
transcribe_output_prefix = 'transcribed'
for key in conn.list_objects_v2(Bucket=bucket, Prefix='input')['Contents']:
    if 'temp' in key['Key']:
        continue
    object_name=key['Key']
    media_input_uri = f's3://{bucket}/{object_name}'

    #create the transcription job
    job_uuid = uuid.uuid1()
    transcribe_job_name = f"transcribe-job-{job_uuid}"
    output_file = object_name.split('.')[0].replace(" ","_")
    transcribe_output_filename = f'{transcribe_output_prefix}-{output_file}.txt'
    output_files.append([transcribe_output_filename,object_name,""])
    print(f'{media_input_uri} transcribed to {transcribe_output_filename}')

    do_transcription_job(transcribe_job_name, media_input_uri, bucket, transcribe_output_filename)

    

s3://c193754a4977429l14270447t1w992382636323-labbucket-8ksrrtliy6u4/input/Mod01_Course Overview.mp4 transcribed to transcribed-input/Mod01_Course_Overview.txt
s3://c193754a4977429l14270447t1w992382636323-labbucket-8ksrrtliy6u4/input/Mod02_Intro.mp4 transcribed to transcribed-input/Mod02_Intro.txt
s3://c193754a4977429l14270447t1w992382636323-labbucket-8ksrrtliy6u4/input/Mod02_Sect01.mp4 transcribed to transcribed-input/Mod02_Sect01.txt
s3://c193754a4977429l14270447t1w992382636323-labbucket-8ksrrtliy6u4/input/Mod02_Sect02.mp4 transcribed to transcribed-input/Mod02_Sect02.txt
s3://c193754a4977429l14270447t1w992382636323-labbucket-8ksrrtliy6u4/input/Mod02_Sect03.mp4 transcribed to transcribed-input/Mod02_Sect03.txt
s3://c193754a4977429l14270447t1w992382636323-labbucket-8ksrrtliy6u4/input/Mod02_Sect04.mp4 transcribed to transcribed-input/Mod02_Sect04.txt
s3://c193754a4977429l14270447t1w992382636323-labbucket-8ksrrtliy6u4/input/Mod02_Sect05.mp4 transcribed to transcribed-input/Mod02_Sect05.t

In [66]:
print(output_files)

[["Hi and welcome to Amazon Academy Machine Learning Foundations. In this module, you'll learn about the course objectives, various job roles in the machine learning domain, and where you can go to learn more about machine learning. After completing this module, you should be able to identify course prerequisites and objectives, indicate the role of the data scientist in business, and identify resources for further learning. We're now going to look at the prerequisites for taking this course. Before you take this course, we recommend that you first complete AWS Academy Cloud Foundations. You should also have some general technical knowledge of IT, including foundational computer literacy skills like basic computer concepts, email, file management, and a good understanding of the internet. We also recommend that you have intermediate skills with Python programming and a general knowledge of applied statistics. Finally, general business knowledge is important for this course. This includ

In [110]:
job=None
while True:
    job = transcribe_client.get_transcription_job(TranscriptionJobName = transcribe_job_name)
    if job['TranscriptionJob']['TranscriptionJobStatus'] in ['COMPLETED','FAILED']:
        break
    print('.', end='')
    sleep(20)
        
print(job['TranscriptionJob']['TranscriptionJobStatus'])

COMPLETED


In [111]:
import boto3, json

# Force a clean session with minimal headers
session = boto3.Session(region_name='us-east-1')
s3_client = session.client('s3', config=boto3.session.Config(
    signature_version='s3v4',
    retries={'max_attempts': 3}
))

transcribed_text = []

for transcribe_output_filename in output_files:
    s3_key = transcribe_output_filename[0]
    
    # Skip entries that were already replaced with text (not a file path)
    if not s3_key.startswith('transcribed'):
        print(f"Skipping (already text or wrong format): {s3_key[:60]}")
        continue
    
    try:
        result = s3_client.get_object(Bucket=bucket, Key=s3_key)
        data = json.loads(result['Body'].read().decode('utf-8'))
        transcription = data['results']['transcripts'][0]['transcript']
        transcribe_output_filename[0] = transcription
        transcribed_text.append(transcription)
        print(f"Loaded: {s3_key[:60]}")
    except Exception as e:
        print(f"FAILED key='{s3_key}' | {type(e).__name__}: {e}")
        break


Skipping (already text or wrong format): Hi and welcome to Amazon Academy Machine Learning Foundation
Skipping (already text or wrong format): Hi and welcome to module 2 of AWS Academy Machine Learning. 
Skipping (already text or wrong format): Hi and welcome to section one. In this section, we're going 
Skipping (already text or wrong format): Hi and welcome back. In this section, we're going to look at
Skipping (already text or wrong format): Hi and welcome back. This is section 3, and we're going to g
Skipping (already text or wrong format): Welcome back. In this section, we'll look at some of the too
Skipping (already text or wrong format): Hi, welcome back. This is section 5, and we're going to disc
Skipping (already text or wrong format): It's now time to review the module. Here are the main takeaw
Skipping (already text or wrong format): Welcome back to AWS Academy of Machine Learning. This is mod
Skipping (already text or wrong format): Hi and welcome back to module 3. This is 

In [112]:
print(output_files[1])

["Hi and welcome to module 2 of AWS Academy Machine Learning. In this module, we're going to introduce machine learning. We'll first look at the business problems that can be solved by machine learning. We'll then talk about terminology, process, tools, and some of the challenges you'll face. After completing this module, you should be able to recognize how machine learning and deep learning are part of artificial intelligence. Describe artificial intelligence and machine learning terminology. Identify how machine learning can be used to solve a business problem. Describe the machine learning process. List the tools available to data scientists. And identify when to use machine learning instead of traditional software development methods. You're now ready to get started with section one. See you in the next video.", 'input/Mod02_Intro.mp4', '']


## 3. Normalizing the text
([Go to top](#Capstone-8:-Bringing-It-All-Together))

Use this section to perform any text normalization steps that are necessary for your solution.

### Reading the dataset:

In [116]:
import pandas as pd

# Rebuild df with correct column names matching what's in output_files
df = pd.DataFrame(output_files, columns=['Transcription', 'Video', 'Transcription_normalized'])

pd.set_option('display.max_colwidth', 150)

In [117]:
df.head()

,Transcription,Video,Transcription_normalized
0,"Hi and welcome to Amazon Academy Machine Learning Foundations. In this module, you'll learn about the course objectives, various job roles in the ...",input/Mod01_Course Overview.mp4,
1,"Hi and welcome to module 2 of AWS Academy Machine Learning. In this module, we're going to introduce machine learning. We'll first look at the bus...",input/Mod02_Intro.mp4,
2,"Hi and welcome to section one. In this section, we're going to talk about what machine learning is. This course is an introduction to machine lear...",input/Mod02_Sect01.mp4,
3,"Hi and welcome back. In this section, we're going to look at the types of business problems machine learning can help you solve. Machine learning ...",input/Mod02_Sect02.mp4,
4,"Hi and welcome back. This is section 3, and we're going to give you a quick high-level overview of machine learning terminology and a typical work...",input/Mod02_Sect03.mp4,


In [118]:
from nltk.corpus import stopwords
from nltk.tokenize import RegexpTokenizer
from nltk.stem.wordnet import WordNetLemmatizer

tokenizer = RegexpTokenizer(r'\w+')
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def normalize_text(text):
    text = text.lower()
    tokens = tokenizer.tokenize(text)
    tokens = [t for t in tokens if t not in stop_words]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return ' '.join(tokens)

# Apply normalization and store back in df
df['Transcription_normalized'] = df['Transcription'].apply(normalize_text)

pd.set_option('display.max_colwidth', 150)
df.head()

,Transcription,Video,Transcription_normalized
0,"Hi and welcome to Amazon Academy Machine Learning Foundations. In this module, you'll learn about the course objectives, various job roles in the ...",input/Mod01_Course Overview.mp4,hi welcome amazon academy machine learning foundation module learn course objective various job role machine learning domain go learn machine lear...
1,"Hi and welcome to module 2 of AWS Academy Machine Learning. In this module, we're going to introduce machine learning. We'll first look at the bus...",input/Mod02_Intro.mp4,hi welcome module 2 aws academy machine learning module going introduce machine learning first look business problem solved machine learning talk ...
2,"Hi and welcome to section one. In this section, we're going to talk about what machine learning is. This course is an introduction to machine lear...",input/Mod02_Sect01.mp4,hi welcome section one section going talk machine learning course introduction machine learning also known ml first discus machine learning fit la...
3,"Hi and welcome back. In this section, we're going to look at the types of business problems machine learning can help you solve. Machine learning ...",input/Mod02_Sect02.mp4,hi welcome back section going look type business problem machine learning help solve machine learning used across digital life email spam filter r...
4,"Hi and welcome back. This is section 3, and we're going to give you a quick high-level overview of machine learning terminology and a typical work...",input/Mod02_Sect03.mp4,hi welcome back section 3 going give quick high level overview machine learning terminology typical workflow cover topic detail later course focus...


## 4. Extracting key phrases and topics
([Go to top](#Capstone-8:-Bringing-It-All-Together))

Use this section to extract the key phrases and topics from the videos.

In [122]:
# uploading normalized transcriptions to S3 before using amazon comprehend...
# Amazon Comprehend --> extract key phrases and topics

import boto3
import uuid
import io
from time import sleep

# Variables del lab - completar con tus valores
bucket = "c193754a4977429l14270447t1w992382636323-labbucket-8ksrrtliy6u4"
job_data_access_role = 'arn:aws:iam::992382636323:role/service-role/c193754a4977429l14270447t1-ComprehendDataAccessRole-e4DbiuHdOqqy'
comprehend_client = boto3.client('comprehend')

comprehend_input_file = 'comprehend_input.csv'
comprehend_prefix = 'comprehend'

def upload_comprehend_input(texts, bucket, prefix, filename):
    csv_buffer = io.StringIO()
    for text in texts:
        line = text.replace('\n', ' ').strip()[:5000]
        csv_buffer.write(line + '\n')
    s3_client = boto3.client('s3')
    s3_client.put_object(
        Bucket=bucket,
        Key=f'{prefix}/{filename}',
        Body=csv_buffer.getvalue()
    )
    return f's3://{bucket}/{prefix}/{filename}'

input_s3_uri = upload_comprehend_input(
    df['Transcription_normalized'].tolist(),  
    bucket,
    comprehend_prefix,
    comprehend_input_file
)
print(f'Input uploaded to: {input_s3_uri}')

Input uploaded to: s3://c193754a4977429l14270447t1w992382636323-labbucket-8ksrrtliy6u4/comprehend/comprehend_input.csv


In [124]:
# Now i'll implement amazon comprehend job, it may take a while so
# i'll do polling after this cell 

input_data_format = 'ONE_DOC_PER_LINE'
output_s3_uri = f's3://{bucket}/comprehend-output/'

# Job 1: Key Phrases
kpe_job_name = f'kpe-job-{uuid.uuid1()}'
kpe_response = comprehend_client.start_key_phrases_detection_job(
    InputDataConfig={
        'S3Uri': input_s3_uri,
        'InputFormat': input_data_format
    },
    OutputDataConfig={'S3Uri': output_s3_uri},
    DataAccessRoleArn=job_data_access_role,
    JobName=kpe_job_name,
    LanguageCode='en'
)
kpe_job_id = kpe_response['JobId']
print(f'Key Phrases job: {kpe_job_id}')

# Job 2: Entities
entity_job_name = f'entity-job-{uuid.uuid1()}'
entity_response = comprehend_client.start_entities_detection_job(
    InputDataConfig={
        'S3Uri': input_s3_uri,
        'InputFormat': input_data_format
    },
    OutputDataConfig={'S3Uri': output_s3_uri},
    DataAccessRoleArn=job_data_access_role,
    JobName=entity_job_name,
    LanguageCode='en'
)
entity_job_id = entity_response['JobId']
print(f'Entities job : {entity_job_id}')

Key Phrases job lanzado: 67d301b3baa55eb40400f3b293b3e883
Entities job lanzado: ab0a3ebe3b35ec187bb2633109cdaa4a


In [126]:
# Polling to check when jobs are finished 

kpe_done = False
entity_done = False

while not (kpe_done and entity_done):
    # Check Key Phrases
    if not kpe_done:
        kpe_job = comprehend_client.describe_key_phrases_detection_job(JobId=kpe_job_id)
        kpe_status = kpe_job['KeyPhrasesDetectionJobProperties']['JobStatus']
        if kpe_status in ['COMPLETED', 'FAILED']:
            kpe_done = True
            print(f'Key Phrases: {kpe_status}')

    # check Entities
    if not entity_done:
        entity_job = comprehend_client.describe_entities_detection_job(JobId=entity_job_id)
        entity_status = entity_job['EntitiesDetectionJobProperties']['JobStatus']
        if entity_status in ['COMPLETED', 'FAILED']:
            entity_done = True
            print(f'Entities: {entity_status}')

    if not (kpe_done and entity_done):
        print('.', end='')
        sleep(30)

print('\n Both jobs done')

.......Key Phrases: COMPLETED
Entities: COMPLETED

 Both jobs done


### Download and parse results:

In [127]:
import tarfile, json

def download_and_parse_comprehend_output(job_properties_key, job_response, output_filename):
    output_uri = job_response[job_properties_key]['OutputDataConfig']['S3Uri']
    out_bucket, out_key = output_uri.replace('s3://', '').split('/', 1)
    
    s3_client = boto3.resource('s3')
    s3_client.meta.client.download_file(out_bucket, out_key, output_filename)
    
    with tarfile.open(output_filename) as tf:
        tf.extractall()
    
    results = []
    with open('output', 'r') as f:
        for line in f:
            results.append(json.loads(line))
    return results

# Downloading key phrases...
kpe_results = download_and_parse_comprehend_output(
    'KeyPhrasesDetectionJobProperties',
    kpe_job,
    'output-kpe.tar.gz'
)

# Downloading Entities...
entity_results = download_and_parse_comprehend_output(
    'EntitiesDetectionJobProperties',
    entity_job,
    'output-entities.tar.gz'
)

print(f'Processed Key phrases documents: {len(kpe_results)}')
print(f'Processed entities (documents)   : {len(entity_results)}')

Processed Key phrases documents: 46
Processed entities (documents)   : 46


/tmp/ipykernel_9153/736880532.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extractall()


## 5. Creating the dashboard
([Go to top](#Capstone-8:-Bringing-It-All-Together))

Use this section to create the dashboard for your solution.

In [136]:
# dependencies

!pip install --upgrade pip
!pip install opensearch-py
!pip install requests
!pip install requests-aws4auth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 124.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [opensearch-py]0m [opensearch-py]


### For dashboard visualization i'll use amazon Kibana:

In [ ]:
# first ill create open search index

my_ip = '<Your IP>'
print(my_ip)

es_client = boto3.client('es')

access_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "",
                "Effect": "Allow",
                "Principal": {
                    "AWS": "*"
                },
                "Action": "es:*",
                "Resource": "*",
                "Condition": { "IpAddress": { "aws:SourceIp": my_ip } }
            }
        ]
    }

190.210.32.235


In [ ]:
# Now i create OpenSearch cluster:

response = es_client.create_elasticsearch_domain(
    DomainName='nlp-lab',
    ElasticsearchVersion='7.9',
    ElasticsearchClusterConfig={
        'InstanceType': 't3.small.elasticsearch', 
        'InstanceCount': 2,
        'DedicatedMasterEnabled': False,
        'ZoneAwarenessEnabled': False
    },
    EBSOptions={
        'EBSEnabled': True,
        'VolumeType': 'gp2',
        'VolumeSize': 10
    },
    AccessPolicies=json.dumps(access_policy)
)

In [144]:

from time import sleep
alive = es_client.describe_elasticsearch_domain(DomainName='nlp-lab')
while alive['DomainStatus']['Processing']:
    print('.', end='')
    sleep(10)
    alive = es_client.describe_elasticsearch_domain(DomainName='nlp-lab')
    
print('ready!')

# Finally creating Kibana dashboard

es_domain = es_client.describe_elasticsearch_domain(DomainName='nlp-lab')
es_endpoint = es_domain['DomainStatus']['Endpoint']
print(f'https://{es_endpoint}/_plugin/kibana')




...........................................ready!
https://search-nlp-lab-3jchgtpjhui2jxvx37ub4dnyfi.us-east-1.es.amazonaws.com/_plugin/kibana


In [145]:
# import opensearch libraries

from opensearchpy import OpenSearch, RequestsHttpConnection
from requests_aws4auth import AWS4Auth
import requests

In [147]:
# creating an opensearch client

from opensearchpy import OpenSearch, RequestsHttpConnection
from requests_aws4auth import AWS4Auth
import boto3

region = 'us-east-1'
credentials = boto3.Session().get_credentials()

awsauth = AWS4Auth(
    credentials.access_key,
    credentials.secret_key,
    region,
    'es',
    session_token=credentials.token
)

es_endpoint = 'search-nlp-lab-3jchgtpjhui2jxvx37ub4dnyfi.us-east-1.es.amazonaws.com'

es = OpenSearch(
    hosts=[{'host': es_endpoint, 'port': 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection
)

# Verify connection
print(es.info())

{'name': 'ba41e56736b8908db01401076a36388c', 'cluster_name': '992382636323:nlp-lab', 'cluster_uuid': 'oo85VtEvTUON3vXO84E4xQ', 'version': {'number': '7.9.1', 'build_flavor': 'oss', 'build_type': 'tar', 'build_hash': 'unknown', 'build_date': '2026-02-06T12:28:17.702161Z', 'build_snapshot': False, 'lucene_version': '8.6.2', 'minimum_wire_compatibility_version': '6.8.0', 'minimum_index_compatibility_version': '6.0.0-beta1'}, 'tagline': 'You Know, for Search'}


In [151]:
# Extraer solo el texto de cada keyphrase y entity
df['KeyPhrases'] = [
    [kp['Text'] for kp in result['KeyPhrases']]
    for result in kpe_results
]

df['Entities'] = [
    [ent['Text'] for ent in result['Entities']]
    for result in entity_results
]

print(df.columns.tolist())
print(df.head(2))

['Transcription', 'Video', 'Transcription_normalized', 'KeyPhrases', 'Entities']
                                                                                                                                           Transcription  \
0  Hi and welcome to Amazon Academy Machine Learning Foundations. In this module, you'll learn about the course objectives, various job roles in the ...   
1  Hi and welcome to module 2 of AWS Academy Machine Learning. In this module, we're going to introduce machine learning. We'll first look at the bus...   

                             Video  \
0  input/Mod01_Course Overview.mp4   
1            input/Mod02_Intro.mp4   

                                                                                                                                Transcription_normalized  \
0  hi welcome amazon academy machine learning foundation module learn course objective various job role machine learning domain go learn machine lear...   
1  hi welcome module 2 

In [152]:
#indexing in opensearch

from time import sleep

index_name = 'videos'

for i, row in df.iterrows():
    document = {
        'video_name' : row['Video'],
        'transcript' : row['Transcription'],
        'normalized' : row['Transcription_normalized'],
        'keyphrases' : row['KeyPhrases'],
        'entities'   : row['Entities'],
    }
    es.index(index=index_name, body=document)

    if i % 10 == 0 and i > 0:
        print(f'Indexed {i} documents...')
        sleep(2)

print(f'Done. Total indexed: {len(df)} documents')

Indexed 10 documents...
Indexed 20 documents...
Indexed 30 documents...
Indexed 40 documents...
Done. Total indexed: 46 documents


# Congratulations!

You have completed this lab, and you can now end the lab by following the lab guide instructions.

*©2023 Amazon Web Services, Inc. or its affiliates. All rights reserved. This work may not be reproduced or redistributed, in whole or in part, without prior written permission from Amazon Web Services, Inc. Commercial copying, lending, or selling is prohibited. All trademarks are the property of their owners.*
